# YOLO11 bottle detector — local training and quality audit

This notebook fine-tunes a **YOLO11 detection model** on
`datasets/wine_bottles_yolo_1000.zip`, validates it, visualizes the best and
worst validation examples, and finally runs an optional confidence-only audit
on the local real-photo pool.

The archive contains five-value YOLO bounding-box annotations, not segmentation
polygons. Therefore the correct base checkpoint is `yolo11n.pt`, not
`yolo11x-seg.pt`. The trained model returns rectangular whole-bottle candidates;
it does not produce pixel masks.

Quality is determined primarily by **Precision, Recall, mAP50, and mAP50-95**.
Average confidence is reported only as a diagnostic because a confidently wrong
detector can still have poor accuracy.

In [ ]:
from __future__ import annotations

import hashlib
import gc
import json
import math
import os
import platform
import random
import shutil
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont, ImageOps
from tqdm.auto import tqdm
from ultralytics import YOLO

# ---------------------------------------------------------------- paths
PROJECT_ROOT = Path("/Users/konstantinmelnikov/Desktop/work/VIscaNEr").resolve()
ARCHIVE_PATH = PROJECT_ROOT / "datasets" / "wine_bottles_yolo_1000.zip"
DATASET_ROOT = PROJECT_ROOT / "datasets" / "wine_bottles_yolo_1000"
REAL_PHOTO_ROOT = PROJECT_ROOT / "datasets" / "real_photos_v4" / "queries"
OUTPUT_ROOT = PROJECT_ROOT / "bottle_reranker" / "outputs" / "yolo_bottle_detector"
MODELS_DIR = PROJECT_ROOT / "models" / "bottle_reranker"
RUNS_DIR = PROJECT_ROOT / "runs" / "bottle_reranker"

# ---------------------------------------------------------- training settings
BASE_MODEL_NAME = "yolo11n.pt"
IMAGE_SIZE = 768
EPOCHS = 80
PATIENCE = 15
SEED = 42
WORKERS = 4 if platform.system() == "Darwin" else 8
RUN_NAME = "yolo11n_wine_bottle_1000"

# --------------------------------------------------------- validation settings
MATCH_IOU = 0.50
AUDIT_CONFIDENCE = 0.25
BEST_EXAMPLES = 12
WORST_EXAMPLES = 24

# Optional domain-shift audit. These photos do not have bottle boxes, so this
# section can rank confidence but cannot measure correctness.
RUN_REAL_PHOTO_AUDIT = True
REAL_MAX_IMAGES: int | None = None  # e.g. 100 for a quick smoke run
REAL_AUDIT_BATCH_SIZE = 2  # intentionally small: 768 px inference can exhaust shared MPS memory
MPS_CACHE_CLEAR_INTERVAL = 8
LOWEST_CONFIDENCE_N = 30
HIGHEST_CONFIDENCE_N = 10
CROP_PADDING = 0.06

if torch.cuda.is_available():
    DEVICE = 0
    DEVICE_NAME = torch.cuda.get_device_name(0)
    BATCH_SIZE = 16
    AMP = True
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    DEVICE_NAME = "Apple Metal (MPS)"
    BATCH_SIZE = 8
    AMP = False
else:
    DEVICE = "cpu"
    DEVICE_NAME = platform.processor() or "CPU"
    BATCH_SIZE = 4
    AMP = False

for required in (PROJECT_ROOT, ARCHIVE_PATH):
    if not required.exists():
        raise FileNotFoundError(required)

for directory in (OUTPUT_ROOT, MODELS_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Python:      {sys.version.split()[0]}")
print(f"PyTorch:     {torch.__version__}")
print(f"Ultralytics: {__import__('ultralytics').__version__}")
print(f"Device:      {DEVICE_NAME} ({DEVICE})")
print(f"Batch size:  {BATCH_SIZE}")
print(f"Archive:     {ARCHIVE_PATH}")
print(f"Output:      {OUTPUT_ROOT}")

## 1. Extract and validate the annotated dataset

Extraction is idempotent and rejects unsafe ZIP paths. The notebook verifies
every image/label pair, class id, normalized coordinate, and annotation line.
It also reconciles the labels with `bottles.csv`.

In [ ]:
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def safe_extract_zip(archive_path: Path, destination: Path) -> None:
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe path in archive: {member.filename}")
        archive.extractall(destination)


if not (DATASET_ROOT / "data.yaml").is_file():
    print(f"Extracting {ARCHIVE_PATH.name}...")
    safe_extract_zip(ARCHIVE_PATH, DATASET_ROOT.parent)
else:
    print(f"Using existing extracted dataset: {DATASET_ROOT}")

required_files = [
    DATASET_ROOT / "data.yaml",
    DATASET_ROOT / "bottles.csv",
    DATASET_ROOT / "build.json",
]
for required in required_files:
    if not required.is_file():
        raise FileNotFoundError(required)


def image_files(split: str) -> list[Path]:
    root = DATASET_ROOT / "images" / split
    return sorted(
        path for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


def validate_split(split: str) -> dict[str, int]:
    images = image_files(split)
    labels_root = DATASET_ROOT / "labels" / split
    labels = sorted(labels_root.glob("*.txt"))
    image_by_stem = {path.stem: path for path in images}
    label_by_stem = {path.stem: path for path in labels}
    missing_labels = sorted(set(image_by_stem) - set(label_by_stem))
    orphan_labels = sorted(set(label_by_stem) - set(image_by_stem))
    errors: list[str] = []
    boxes = 0
    empty_labels = 0

    for label_path in labels:
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
        if not lines:
            empty_labels += 1
        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            if len(parts) != 5:
                errors.append(f"{label_path}:{line_number}: expected 5 values")
                continue
            try:
                class_id = int(parts[0])
                x_center, y_center, width, height = map(float, parts[1:])
            except ValueError:
                errors.append(f"{label_path}:{line_number}: non-numeric value")
                continue
            if class_id != 0:
                errors.append(f"{label_path}:{line_number}: class {class_id}, expected 0")
            if not all(0.0 <= value <= 1.0 for value in (x_center, y_center, width, height)):
                errors.append(f"{label_path}:{line_number}: coordinate outside [0, 1]")
            if width <= 0 or height <= 0:
                errors.append(f"{label_path}:{line_number}: non-positive box size")
            boxes += 1

    if missing_labels or orphan_labels or errors:
        raise RuntimeError(
            f"Invalid {split} split: missing_labels={missing_labels[:5]}, "
            f"orphan_labels={orphan_labels[:5]}, errors={errors[:10]}"
        )
    return {
        "images": len(images),
        "labels": len(labels),
        "boxes": boxes,
        "empty_labels": empty_labels,
    }


split_validation = {split: validate_split(split) for split in ("train", "val")}
bottles = pd.read_csv(DATASET_ROOT / "bottles.csv")
bottles["whole_bottle"] = bottles["whole_bottle"].astype(str).str.lower().eq("true")
bottles["truncated_by_frame"] = bottles["truncated_by_frame"].astype(str).str.lower().eq("true")
build_info = json.loads((DATASET_ROOT / "build.json").read_text())

if sum(item["boxes"] for item in split_validation.values()) != len(bottles):
    raise RuntimeError("YOLO label count does not match bottles.csv")
reported_counts = build_info.get("counts", {})
if reported_counts.get("boxes_total") != len(bottles):
    raise RuntimeError("build.json box count does not match bottles.csv")
if reported_counts.get("whole_bottles") != int(bottles["whole_bottle"].sum()):
    raise RuntimeError("build.json whole-bottle count does not match bottles.csv")

# Use an absolute path so execution does not depend on the notebook cwd.
LOCAL_DATA_YAML = OUTPUT_ROOT / "data.local.yaml"
LOCAL_DATA_YAML.write_text(
    yaml.safe_dump(
        {
            "path": str(DATASET_ROOT),
            "train": "images/train",
            "val": "images/val",
            "names": {0: "bottle"},
        },
        sort_keys=False,
    )
)

dataset_summary = pd.DataFrame(
    [
        {"split": split, **values}
        for split, values in split_validation.items()
    ]
)
display(dataset_summary)
print(f"Annotated boxes:   {len(bottles):,}")
print(f"Whole bottles:     {int(bottles['whole_bottle'].sum()):,}")
print(f"Frame-truncated:   {int(bottles['truncated_by_frame'].sum()):,}")
print(f"Unique identities: {bottles['wine_slug'].nunique():,}")
print(f"Runtime YAML:      {LOCAL_DATA_YAML}")

In [ ]:
def load_rgb(path: Path) -> Image.Image:
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert("RGB")


def draw_ground_truth(image_path: Path, rows: pd.DataFrame) -> Image.Image:
    image = load_rgb(image_path)
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()
    for row in rows.itertuples(index=False):
        box = tuple(int(getattr(row, key)) for key in ("padded_x1", "padded_y1", "padded_x2", "padded_y2"))
        color = "#16a34a" if row.whole_bottle else "#f59e0b"
        draw.rectangle(box, outline=color, width=4)
        draw.text((box[0] + 4, box[1] + 4), "whole" if row.whole_bottle else "truncated", fill=color, font=font)
    return image


def make_image_grid(images: list[tuple[Image.Image, str]], destination: Path, columns: int = 4) -> Path:
    tile_width, tile_height, caption_height = 320, 380, 70
    if not images:
        sheet = Image.new("RGB", (640, 160), "white")
        ImageDraw.Draw(sheet).text((20, 70), "No examples for this category", fill="black", font=ImageFont.load_default())
        sheet.save(destination, quality=92, subsampling=0)
        return destination
    rows_count = math.ceil(len(images) / columns)
    sheet = Image.new("RGB", (columns * tile_width, rows_count * (tile_height + caption_height)), "white")
    draw = ImageDraw.Draw(sheet)
    font = ImageFont.load_default()
    for index, (image, caption) in enumerate(images):
        image = image.copy()
        image.thumbnail((tile_width - 12, tile_height - 12), Image.Resampling.LANCZOS)
        x0 = (index % columns) * tile_width
        y0 = (index // columns) * (tile_height + caption_height)
        sheet.paste(image, (x0 + (tile_width - image.width) // 2, y0 + (tile_height - image.height) // 2))
        draw.multiline_text((x0 + 6, y0 + tile_height + 4), caption[:150], fill="black", font=font, spacing=3)
    sheet.save(destination, quality=92, subsampling=0)
    return destination


preview_paths = random.Random(SEED).sample(image_files("val"), k=min(12, len(image_files("val"))))
preview_tiles = []
for image_path in preview_paths:
    relative = f"images/val/{image_path.name}"
    rows = bottles[bottles["image"].eq(relative)]
    preview_tiles.append((draw_ground_truth(image_path, rows), f"GT: {image_path.name}\nboxes={len(rows)}"))

gt_preview = make_image_grid(preview_tiles, OUTPUT_ROOT / "ground_truth_preview.jpg")
display(Image.open(gt_preview))

## 2. Fine-tune YOLO11n

The detector is initialized from the original COCO-pretrained `yolo11n.pt`.
Augmentations are intentionally moderate; vertical and horizontal flips are
disabled because label/capsule text should not be mirrored. Training writes a
new Ultralytics run directory and copies the best checkpoint to the stable
project model path.

In [ ]:
def load_base_model(model_name: str) -> YOLO:
    local_weights = MODELS_DIR / model_name
    if local_weights.is_file():
        return YOLO(str(local_weights))
    previous_cwd = Path.cwd()
    try:
        os.chdir(MODELS_DIR)
        model = YOLO(model_name)
    finally:
        os.chdir(previous_cwd)
    return model


TRAIN_CONFIG = {
    "data": str(LOCAL_DATA_YAML),
    "imgsz": IMAGE_SIZE,
    "epochs": EPOCHS,
    "batch": BATCH_SIZE,
    "patience": PATIENCE,
    "device": DEVICE,
    "workers": WORKERS,
    "project": str(RUNS_DIR),
    "name": RUN_NAME,
    "exist_ok": False,
    "pretrained": True,
    "optimizer": "auto",
    "seed": SEED,
    "deterministic": True,
    "amp": AMP,
    "cache": False,
    "plots": True,
    "degrees": 7.0,
    "translate": 0.08,
    "scale": 0.25,
    "perspective": 0.0005,
    "hsv_h": 0.01,
    "hsv_s": 0.35,
    "hsv_v": 0.25,
    "mosaic": 0.20,
    "close_mosaic": 10,
    "fliplr": 0.0,
    "flipud": 0.0,
}

print(json.dumps({key: str(value) for key, value in TRAIN_CONFIG.items()}, indent=2))
model = load_base_model(BASE_MODEL_NAME)
train_result = model.train(**TRAIN_CONFIG)

TRAIN_SAVE_DIR = Path(model.trainer.save_dir).resolve()
source_best = TRAIN_SAVE_DIR / "weights" / "best.pt"
source_last = TRAIN_SAVE_DIR / "weights" / "last.pt"
if not source_best.is_file():
    raise FileNotFoundError(source_best)

BEST_MODEL_PATH = MODELS_DIR / "best_bottle_detector.pt"
LAST_MODEL_PATH = MODELS_DIR / "last_bottle_detector.pt"
shutil.copy2(source_best, BEST_MODEL_PATH)
if source_last.is_file():
    shutil.copy2(source_last, LAST_MODEL_PATH)

print(f"Run directory: {TRAIN_SAVE_DIR}")
print(f"Best model:    {BEST_MODEL_PATH}")
print(f"Last model:    {LAST_MODEL_PATH if LAST_MODEL_PATH.is_file() else 'not created'}")

## 3. Official validation metrics

- **Precision**: how many predicted boxes are correct.
- **Recall**: how many annotated bottles were found.
- **mAP50**: average precision at IoU 0.50.
- **mAP50-95**: stricter localization quality averaged over IoU 0.50–0.95.

For a crop generator, Recall and mAP50-95 matter more than average confidence:
missing a bottle or cutting its neck cannot be repaired by a confident score.

In [ ]:
best_model = YOLO(str(BEST_MODEL_PATH))
val_metrics = best_model.val(
    data=str(LOCAL_DATA_YAML),
    split="val",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=WORKERS,
    conf=0.001,
    iou=0.70,
    plots=True,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_validation",
    exist_ok=False,
    verbose=False,
)

precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)
official_metrics = {
    "precision": precision,
    "recall": recall,
    "f1": 2 * precision * recall / max(precision + recall, 1e-12),
    "map50": float(val_metrics.box.map50),
    "map50_95": float(val_metrics.box.map),
    "fitness": float(val_metrics.fitness),
}
(OUTPUT_ROOT / "official_validation_metrics.json").write_text(json.dumps(official_metrics, indent=2))
display(pd.DataFrame([official_metrics]).style.format("{:.4f}"))

plot_candidates = [
    TRAIN_SAVE_DIR / "results.png",
    Path(val_metrics.save_dir) / "PR_curve.png",
    Path(val_metrics.save_dir) / "F1_curve.png",
    Path(val_metrics.save_dir) / "confusion_matrix_normalized.png",
]
for plot_path in plot_candidates:
    if plot_path.is_file():
        print(plot_path)
        display(Image.open(plot_path))

In [ ]:
def pairwise_iou(gt_boxes: np.ndarray, pred_boxes: np.ndarray) -> np.ndarray:
    if len(gt_boxes) == 0 or len(pred_boxes) == 0:
        return np.zeros((len(gt_boxes), len(pred_boxes)), dtype=np.float32)
    top_left = np.maximum(gt_boxes[:, None, :2], pred_boxes[None, :, :2])
    bottom_right = np.minimum(gt_boxes[:, None, 2:], pred_boxes[None, :, 2:])
    intersection = np.clip(bottom_right - top_left, 0, None)
    intersection_area = intersection[..., 0] * intersection[..., 1]
    gt_area = np.prod(np.clip(gt_boxes[:, 2:] - gt_boxes[:, :2], 0, None), axis=1)
    pred_area = np.prod(np.clip(pred_boxes[:, 2:] - pred_boxes[:, :2], 0, None), axis=1)
    union = gt_area[:, None] + pred_area[None, :] - intersection_area
    return intersection_area / np.maximum(union, 1e-9)


def greedy_match(iou_matrix: np.ndarray, threshold: float) -> list[tuple[int, int, float]]:
    candidates = [
        (float(iou_matrix[gt_index, pred_index]), gt_index, pred_index)
        for gt_index in range(iou_matrix.shape[0])
        for pred_index in range(iou_matrix.shape[1])
        if iou_matrix[gt_index, pred_index] >= threshold
    ]
    matches: list[tuple[int, int, float]] = []
    used_gt: set[int] = set()
    used_pred: set[int] = set()
    for iou, gt_index, pred_index in sorted(candidates, reverse=True):
        if gt_index in used_gt or pred_index in used_pred:
            continue
        used_gt.add(gt_index)
        used_pred.add(pred_index)
        matches.append((gt_index, pred_index, iou))
    return matches


val_images = image_files("val")
prediction_results = best_model.predict(
    source=[str(path) for path in val_images],
    imgsz=IMAGE_SIZE,
    conf=AUDIT_CONFIDENCE,
    iou=0.70,
    max_det=60,
    device=DEVICE,
    half=AMP,
    verbose=False,
)

prediction_by_name = {}
image_rows: list[dict] = []
object_rows: list[dict] = []

for image_path, result in zip(val_images, prediction_results, strict=True):
    relative = f"images/val/{image_path.name}"
    gt_rows = bottles[bottles["image"].eq(relative)].reset_index(drop=True)
    gt_boxes = gt_rows[["padded_x1", "padded_y1", "padded_x2", "padded_y2"]].to_numpy(dtype=np.float32)
    if result.boxes is None or len(result.boxes) == 0:
        pred_boxes = np.empty((0, 4), dtype=np.float32)
        pred_conf = np.empty((0,), dtype=np.float32)
    else:
        pred_boxes = result.boxes.xyxy.detach().float().cpu().numpy()
        pred_conf = result.boxes.conf.detach().float().cpu().numpy()

    matches = greedy_match(pairwise_iou(gt_boxes, pred_boxes), MATCH_IOU)
    matched_gt = {gt_index for gt_index, _, _ in matches}
    matched_pred = {pred_index for _, pred_index, _ in matches}
    match_by_gt = {gt_index: (pred_index, iou) for gt_index, pred_index, iou in matches}
    true_positive = len(matches)
    false_negative = len(gt_boxes) - true_positive
    false_positive = len(pred_boxes) - true_positive
    image_precision = true_positive / max(len(pred_boxes), 1)
    image_recall = true_positive / max(len(gt_boxes), 1)
    image_f1 = 2 * image_precision * image_recall / max(image_precision + image_recall, 1e-12)
    mean_iou = float(np.mean([match[2] for match in matches])) if matches else 0.0
    mean_confidence = float(np.mean([pred_conf[index] for index in matched_pred])) if matched_pred else 0.0

    image_rows.append(
        {
            "image": relative,
            "gt_count": len(gt_boxes),
            "prediction_count": len(pred_boxes),
            "tp": true_positive,
            "fn": false_negative,
            "fp": false_positive,
            "image_precision": image_precision,
            "image_recall": image_recall,
            "image_f1": image_f1,
            "mean_matched_iou": mean_iou,
            "mean_matched_confidence": mean_confidence,
            # Transparent audit ranking only; not an official YOLO metric.
            "audit_quality_score": image_f1 * mean_iou,
        }
    )

    for gt_index, gt_row in gt_rows.iterrows():
        match = match_by_gt.get(gt_index)
        object_rows.append(
            {
                "image": relative,
                "instance_index": int(gt_row["instance_index"]),
                "role": gt_row["role"],
                "whole_bottle": bool(gt_row["whole_bottle"]),
                "truncated_by_frame": bool(gt_row["truncated_by_frame"]),
                "matched_at_iou50": match is not None,
                "matched_iou": float(match[1]) if match is not None else np.nan,
                "matched_confidence": float(pred_conf[match[0]]) if match is not None else np.nan,
            }
        )

    prediction_by_name[image_path.name] = {
        "boxes": pred_boxes,
        "confidences": pred_conf,
        "matched_pred": matched_pred,
    }

image_audit = pd.DataFrame(image_rows)
object_audit = pd.DataFrame(object_rows)
image_audit.to_csv(OUTPUT_ROOT / "validation_per_image.csv", index=False)
object_audit.to_csv(OUTPUT_ROOT / "validation_per_object.csv", index=False)


def group_recall(frame: pd.DataFrame) -> float:
    return float(frame["matched_at_iou50"].mean()) if len(frame) else float("nan")


diagnostic_metrics = {
    "audit_confidence_threshold": AUDIT_CONFIDENCE,
    "match_iou_threshold": MATCH_IOU,
    "validation_images": int(len(image_audit)),
    "validation_objects": int(len(object_audit)),
    "whole_bottle_recall_at_iou50": group_recall(object_audit[object_audit["whole_bottle"]]),
    "truncated_bottle_recall_at_iou50": group_recall(object_audit[object_audit["truncated_by_frame"]]),
    "mean_matched_iou": float(object_audit["matched_iou"].mean()),
    "mean_true_positive_confidence": float(object_audit["matched_confidence"].mean()),
    "false_negatives": int(image_audit["fn"].sum()),
    "false_positives": int(image_audit["fp"].sum()),
    "images_with_no_detection": int((image_audit["prediction_count"] == 0).sum()),
}
display(pd.DataFrame([diagnostic_metrics]).T.rename(columns={0: "value"}))

In [ ]:
def draw_validation_overlay(image_path: Path) -> Image.Image:
    relative = f"images/val/{image_path.name}"
    gt_rows = bottles[bottles["image"].eq(relative)]
    prediction = prediction_by_name[image_path.name]
    image = load_rgb(image_path)
    draw = ImageDraw.Draw(image)
    font = ImageFont.load_default()

    for row in gt_rows.itertuples(index=False):
        box = tuple(int(getattr(row, key)) for key in ("padded_x1", "padded_y1", "padded_x2", "padded_y2"))
        color = "#16a34a" if row.whole_bottle else "#f59e0b"
        draw.rectangle(box, outline=color, width=4)
        draw.text((box[0] + 3, box[1] + 3), "GT whole" if row.whole_bottle else "GT truncated", fill=color, font=font)

    for pred_index, (box, confidence) in enumerate(zip(prediction["boxes"], prediction["confidences"], strict=True)):
        x1, y1, x2, y2 = map(int, box)
        matched = pred_index in prediction["matched_pred"]
        color = "#2563eb" if matched else "#dc2626"
        draw.rectangle((x1, y1, x2, y2), outline=color, width=3)
        draw.text((x1 + 3, max(0, y1 - 13)), f"P {confidence:.2f}", fill=color, font=font)
    return image


best_rows = image_audit.sort_values(
    ["audit_quality_score", "fn", "fp", "mean_matched_confidence"],
    ascending=[False, True, True, False],
).head(BEST_EXAMPLES)
worst_rows = image_audit.sort_values(
    ["fn", "fp", "audit_quality_score", "mean_matched_confidence"],
    ascending=[False, False, True, True],
).head(WORST_EXAMPLES)


def audit_tiles(rows: pd.DataFrame) -> list[tuple[Image.Image, str]]:
    tiles = []
    for row in rows.itertuples(index=False):
        image_path = DATASET_ROOT / row.image
        caption = (
            f"{image_path.name}\nTP/FN/FP={row.tp}/{row.fn}/{row.fp} "
            f"IoU={row.mean_matched_iou:.3f} conf={row.mean_matched_confidence:.3f}"
        )
        tiles.append((draw_validation_overlay(image_path), caption))
    return tiles


best_sheet = make_image_grid(audit_tiles(best_rows), OUTPUT_ROOT / "validation_best.jpg")
worst_sheet = make_image_grid(audit_tiles(worst_rows), OUTPUT_ROOT / "validation_worst.jpg")
best_rows.to_csv(OUTPUT_ROOT / "validation_best.csv", index=False)
worst_rows.to_csv(OUTPUT_ROOT / "validation_worst.csv", index=False)

print("Legend: green=whole GT, orange=truncated GT, blue=matched prediction, red=false positive")
display(best_rows[["image", "tp", "fn", "fp", "mean_matched_iou", "mean_matched_confidence"]])
display(Image.open(best_sheet))
display(worst_rows[["image", "tp", "fn", "fp", "mean_matched_iou", "mean_matched_confidence"]])
display(Image.open(worst_sheet))

## 4. Optional real-photo confidence audit

This preserves the original notebook's purpose. It runs the fine-tuned detector
on real photos and exports the lowest- and highest-confidence bottle crops.
There are no bottle-box labels for this pool, so **low/high confidence is not a
measured error/correctness ranking**. Use the supervised validation sheets above
for actual quality judgments.

In [ ]:
def discover_real_images() -> list[Path]:
    paths = sorted(
        path for path in REAL_PHOTO_ROOT.rglob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )
    return paths if REAL_MAX_IMAGES is None else paths[:REAL_MAX_IMAGES]


def padded_box(box: np.ndarray, width: int, height: int, fraction: float) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = map(float, box)
    padding = max(x2 - x1, y2 - y1) * fraction
    return (
        max(0, int(math.floor(x1 - padding))),
        max(0, int(math.floor(y1 - padding))),
        min(width, int(math.ceil(x2 + padding))),
        min(height, int(math.ceil(y2 + padding))),
    )


real_audit_summary = None
if RUN_REAL_PHOTO_AUDIT:
    if not REAL_PHOTO_ROOT.is_dir():
        raise FileNotFoundError(REAL_PHOTO_ROOT)
    real_paths = discover_real_images()
    real_rows = []

    # Do not pass the complete path list to one long MPS stream. Ultralytics can
    # retain decoded tensors/results long enough to exhaust Apple shared memory.
    # Small independent batches keep peak memory bounded.
    # A previously failed streaming run may still be referenced by the notebook
    # kernel. Drop it before asking MPS to release cached allocations.
    globals().pop("real_results", None)
    globals().pop("batch_results", None)
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()

    batch_starts = range(0, len(real_paths), REAL_AUDIT_BATCH_SIZE)
    for batch_number, batch_start in enumerate(
        tqdm(batch_starts, total=math.ceil(len(real_paths) / REAL_AUDIT_BATCH_SIZE), desc="real audit")
    ):
        batch_paths = real_paths[batch_start : batch_start + REAL_AUDIT_BATCH_SIZE]
        batch_results = best_model.predict(
            source=[str(path) for path in batch_paths],
            imgsz=IMAGE_SIZE,
            conf=0.001,
            iou=0.70,
            max_det=60,
            batch=REAL_AUDIT_BATCH_SIZE,
            device=DEVICE,
            half=AMP,
            stream=False,
            verbose=False,
        )

        for image_path, result in zip(batch_paths, batch_results, strict=True):
            if result.boxes is None or len(result.boxes) == 0:
                real_rows.append({"image": str(image_path.relative_to(PROJECT_ROOT)), "status": "no_detection"})
                continue
            confidences = result.boxes.conf.detach().float().cpu().numpy()
            boxes_xyxy = result.boxes.xyxy.detach().float().cpu().numpy()
            selected_index = int(np.argmax(confidences))
            with load_rgb(image_path) as image:
                x1, y1, x2, y2 = padded_box(
                    boxes_xyxy[selected_index],
                    image.width,
                    image.height,
                    CROP_PADDING,
                )
            real_rows.append(
                {
                    "image": str(image_path.relative_to(PROJECT_ROOT)),
                    "status": "detected",
                    "detections": int(len(confidences)),
                    "confidence": float(confidences[selected_index]),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                }
            )

        del batch_results
        if DEVICE == "mps" and (batch_number + 1) % MPS_CACHE_CLEAR_INTERVAL == 0:
            gc.collect()
            torch.mps.empty_cache()

    real_audit = pd.DataFrame(real_rows)
    real_audit.to_csv(OUTPUT_ROOT / "real_photo_predictions.csv", index=False)
    detected = real_audit[real_audit["status"].eq("detected")].copy()
    lowest = detected.nsmallest(min(LOWEST_CONFIDENCE_N, len(detected)), "confidence")
    highest = detected.nlargest(min(HIGHEST_CONFIDENCE_N, len(detected)), "confidence")

    def real_crop_tiles(rows: pd.DataFrame) -> list[tuple[Image.Image, str]]:
        tiles = []
        for row in rows.itertuples(index=False):
            source = PROJECT_ROOT / row.image
            image = load_rgb(source)
            crop = image.crop((int(row.x1), int(row.y1), int(row.x2), int(row.y2)))
            tiles.append((crop, f"conf={row.confidence:.4f} detections={row.detections}\n{source.name}"))
        return tiles

    lowest_sheet = make_image_grid(real_crop_tiles(lowest), OUTPUT_ROOT / "real_lowest_confidence.jpg", columns=5)
    highest_sheet = make_image_grid(real_crop_tiles(highest), OUTPUT_ROOT / "real_highest_confidence.jpg", columns=5)
    real_audit_summary = {
        "images": int(len(real_audit)),
        "detected": int(len(detected)),
        "no_detection": int((real_audit["status"] == "no_detection").sum()),
        "mean_top1_confidence": float(detected["confidence"].mean()) if len(detected) else None,
    }
    print(json.dumps(real_audit_summary, indent=2))
    display(Image.open(lowest_sheet))
    display(Image.open(highest_sheet))
else:
    print("Real-photo audit disabled")

In [ ]:
final_summary = {
    "dataset": {
        "train_images": split_validation["train"]["images"],
        "val_images": split_validation["val"]["images"],
        "annotated_boxes": int(len(bottles)),
        "whole_bottles": int(bottles["whole_bottle"].sum()),
        "truncated_bottles": int(bottles["truncated_by_frame"].sum()),
    },
    "official_validation": official_metrics,
    "diagnostic_validation": diagnostic_metrics,
    "real_photo_confidence_audit": real_audit_summary,
    "best_model": str(BEST_MODEL_PATH),
    "training_run": str(TRAIN_SAVE_DIR),
}
(OUTPUT_ROOT / "final_summary.json").write_text(json.dumps(final_summary, indent=2))

print(json.dumps(final_summary, indent=2))
print("\nInterpretation:")
if official_metrics["recall"] < 0.90:
    print("- Recall < 0.90: inspect missed examples and add real-photo bottle boxes.")
if official_metrics["map50"] - official_metrics["map50_95"] > 0.25:
    print("- Large mAP50 vs mAP50-95 gap: detections exist, but box localization is unstable.")
if diagnostic_metrics["whole_bottle_recall_at_iou50"] < 0.90:
    print("- Whole-bottle recall < 0.90: the model is not yet reliable enough for reranker crops.")
print("- Real-photo confidence is not accuracy; manually inspect the low-confidence contact sheet.")
print(f"- Best checkpoint: {BEST_MODEL_PATH}")
print(f"- All audit artifacts: {OUTPUT_ROOT}")

## Outputs

- Model: `models/bottle_reranker/best_bottle_detector.pt`
- Ultralytics runs: `runs/bottle_reranker/`
- Metrics and validation tables: `bottle_reranker/outputs/yolo_bottle_detector/`
- Best/worst supervised examples: `validation_best.jpg`, `validation_worst.jpg`
- Real-photo confidence extremes: `real_lowest_confidence.jpg`, `real_highest_confidence.jpg`

The annotated dataset is synthetic. Even excellent validation mAP does not prove
production quality on phone photographs; the real-photo contact sheets are the
next manual audit, and a genuinely labeled real-photo holdout is ultimately
required.